# Sequential quadratic programming

SQP solves a constrained problem as a sequence of quadratic programs, each
carrying the *linearized* constraints explicitly rather than penalizing them
away. It is the method behind `scipy.optimize.minimize(method="SLSQP")`.

Three pieces make it work away from the solution: a Powell-damped BFGS model
that stays positive definite where the Lagrangian is not convex, an L1
exact-penalty merit function for the line search, and multipliers re-estimated
from each subproblem to drive the KKT test.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from mopt.nonlinear import (ConstrainedNLPProblem, SQP, damped_bfgs,
                            kkt_residual, lagrangian)

## Test problem

$$\min_x\; (x^T Q x + b^T x)^2 \quad\text{s.t.}\quad x^T M x \le 26$$

In [2]:
Q = np.array([
    [1.7, -0.8, -0.3, 0.7, -0.3],
    [-0.8, 5.0, 1.7, -0.5, -1.5],
    [-0.3, 1.7, 2.2, 0.1, -1.1],
    [0.7, -0.5, 0.1, 3.1, -1.2],
    [-0.3, -1.5, -1.1, -1.2, 2.8],
])
M = np.array([
    [5.5, 1.3, -0.7, 0.0, 1.0],
    [1.3, 3.1, -1.3, 1.4, -0.2],
    [-0.7, -1.3, 5.1, 1.4, -1.5],
    [0.0, 1.4, 1.4, 3.7, -1.9],
    [1.0, -0.2, -1.5, -1.9, 4.8],
])
b = np.array([3.0, -5.0, -5.0, -1.0, 1.0])

f = lambda x: float((x @ Q @ x + b @ x) ** 2)
grad_f = lambda x: 2.0 * (2.0 * (Q @ x) + b) * (x @ Q @ x + b @ x)

def make_problem(x0):
    return ConstrainedNLPProblem(
        f=f, x0=np.asarray(x0, dtype=float), grad=grad_f,
        ineq=lambda x: np.array([x @ M @ x - 26.0]),
        ineq_jac=lambda x: (2.0 * (M @ x))[None, :],
    )

## From many starting points

`x0` need not be feasible — SQP is free to approach from outside, unlike the
barrier method.

In [3]:
rng = np.random.default_rng(42)
inits = np.vstack([np.ones(5), rng.uniform(-2.0, 2.0, size=(10, 5))])

rows = []
for x0 in inits:
    problem = make_problem(x0)
    ref = minimize(f, x0, jac=grad_f, method="SLSQP", tol=1e-12,
                   constraints=[{"type": "ineq", "fun": lambda z: -problem.ineq(z)}])
    result = SQP().solve(problem)
    assert result.success, result.message
    rows.append({
        "x0": tuple(np.round(x0, 3)),
        "iters": result.n_iter,
        "f": result.fun,
        "g(x)": float(problem.ineq(result.x)[0]),
        "KKT": kkt_residual(problem, result.x).max_violation,
        "f (scipy)": ref.fun,
    })

pd.DataFrame(rows).set_index("x0")

,iters,f,g(x),KKT,f (scipy)
x0,,,,,
"(1.0, 1.0, 1.0, 1.0, 1.0)",5,2.777152e-15,-8.424699,5.287858e-07,3.403401e-18
"(1.096, -0.244, 1.434, 0.789, -1.623)",6,2.669212e-17,-20.125285,8.138171e-08,8.322348e-15
"(1.902, 1.045, 1.144, -1.488, -0.198)",9,7.067406e-21,-19.692068,1.618445e-09,4.218356e-14
"(-0.517, 1.707, 0.575, 1.291, -0.226)",9,4.896911e-24,-19.229912,2.852876e-11,6.680504e-19
"(-1.091, 0.218, -1.745, 1.311, 0.527)",8,4.156036e-20,-16.728425,2.758135e-09,2.399796e-13
"(1.032, -0.582, 1.883, 1.572, 1.114)",10,7.462181e-20,-15.634265,5.093999e-09,3.902718e-15
"(-1.221, -0.133, -1.825, -1.383, 0.732)",11,6.949095e-24,-8.441537,5.392324e-11,2.894245e-20
"(0.979, 1.87, -0.697, -0.518, -0.122)",7,3.262593e-16,-7.027885,3.086546e-07,5.671932e-14
"(-1.242, -1.48, -0.097, -1.092, 0.679)",7,2.727089e-20,-5.780834,3.091381e-09,8.591242e-20


## A converged run carries a certificate

`kkt_residual` reports all four KKT conditions separately, so "converged" is
checkable rather than asserted.

In [4]:
problem = make_problem(np.ones(5))
result = SQP().solve(problem)
residual = kkt_residual(problem, result.x)

print(f"stationarity       = {residual.stationarity:.3e}")
print(f"primal feasibility = {residual.primal_feasibility:.3e}")
print(f"dual feasibility   = {residual.dual_feasibility:.3e}")
print(f"complementarity    = {residual.complementarity:.3e}")
print(f"multipliers lambda = {residual.lam}")
print(f"constraint active? = {bool(residual.active[0])}")
print(f"satisfied(1e-5)    = {residual.satisfied(1e-5)}")

stationarity       = 5.288e-07
primal feasibility = 0.000e+00
dual feasibility   = 0.000e+00
complementarity    = 0.000e+00
multipliers lambda = [0.]
constraint active? = False
satisfied(1e-5)    = True


## Equality constraints, and both together

The same solver handles equalities and any mix of the two.

In [5]:
cases = {
    "equality only: min |x|^2 s.t. x0 + x1 = 2": ConstrainedNLPProblem(
        f=lambda x: float(x @ x), x0=np.array([3.0, -1.0]), grad=lambda x: 2.0 * x,
        eq=lambda x: np.array([x[0] + x[1] - 2.0]),
        eq_jac=lambda x: np.array([[1.0, 1.0]])),
    "mixed: min |x-(2,2)|^2 s.t. |x| <= 1, x0 = x1": ConstrainedNLPProblem(
        f=lambda x: float((x[0] - 2.0) ** 2 + (x[1] - 2.0) ** 2),
        x0=np.zeros(2), grad=lambda x: 2.0 * (x - np.array([2.0, 2.0])),
        ineq=lambda x: np.array([x @ x - 1.0]), ineq_jac=lambda x: 2.0 * x[None, :],
        eq=lambda x: np.array([x[0] - x[1]]), eq_jac=lambda x: np.array([[1.0, -1.0]])),
}
for name, p in cases.items():
    r = SQP().solve(p)
    print(f"{name}\n  x = {np.round(r.x, 8)}  iters = {r.n_iter}  "
          f"KKT = {kkt_residual(p, r.x).max_violation:.2e}\n")

equality only: min |x|^2 s.t. x0 + x1 = 2
  x = [1. 1.]  iters = 1  KKT = 5.00e-14

mixed: min |x-(2,2)|^2 s.t. |x| <= 1, x0 = x1
  x = [0.70710678 0.70710678]  iters = 5  KKT = 4.29e-12



## The damped BFGS update

Plain BFGS keeps the model positive definite only when $s^T y > 0$. Minimizing
a Lagrangian gives no such guarantee, so Powell's damping interpolates $y$
toward $Bs$ until the curvature condition holds. When curvature is already
good it reduces to plain BFGS exactly.

In [6]:
B = np.eye(3)
s = np.array([1.0, 0.0, 0.0])

good = damped_bfgs(B, s, np.array([2.0, 0.0, 0.0]))    # s @ y = 2 > 0
print("good curvature -> secant equation B@s == y holds:",
      np.allclose(good @ s, [2.0, 0.0, 0.0]))

bad = damped_bfgs(B, s, np.array([-5.0, 0.0, 0.0]))    # s @ y = -5 < 0
print("negative curvature -> eigenvalues:", np.round(np.linalg.eigvalsh(bad), 4))
print("still positive definite:", np.linalg.eigvalsh(bad).min() > 0)

good curvature -> secant equation B@s == y holds: True
negative curvature -> eigenvalues: [0.2 1.  1. ]
still positive definite: True
